# Min-K% Prob Contamination Probe (open-weight models)

Shi et al. (2024), *Detecting Pretraining Data from LLMs*: for a text, take the k% lowest per-token
log-probabilities under the model and average them. Higher (less negative) → more likely the model has
seen this text before; lower → more likely unseen. Needs raw token log-probabilities, so this only works
for models you can get logprobs from — your local Ollama models, not the closed APIs.

Uses the same trick as a teacher-forced log-likelihood read: ask for `max_tokens=0` with `echo=True` and
`logprobs` on Ollama's OpenAI-compatible `/v1/completions` endpoint, which scores the input text itself
instead of generating anything new.

Three puzzle groups: `success` (10, original+adapted), `partial_fail` (1, original+adapted — a coherent
but judge-rejected rewrite), `empty_fail` (10, original only — nothing to score on the adapted side).

**Before running:** put `contamination_dataset.csv` in the same folder, and make sure the three models
below are pulled locally (`ollama list` to check, `ollama pull <name>` if not — match the tags you
actually run with, these are just the names from your paper's Table 1).

In [ ]:
from openai import OpenAI
import pandas as pd

OLLAMA_URL = "http://localhost:11434/v1"
ollama_client = OpenAI(base_url=OLLAMA_URL, api_key="local")

MODELS = ["qwen3.5:9b", "llama3.1:8b", "deepseek-r1:7b"]   # adjust tags to match `ollama list`
K_PCT = 20   # bottom-k% of tokens, per Shi et al.'s default

## Load the same 21 puzzles

In [ ]:
import csv

CSV_PATH = "contamination_dataset.csv"   # puzzle_id, outcome, original_text, adapted_text

with open(CSV_PATH, newline="", encoding="utf-8-sig") as f:
    rows = list(csv.DictReader(f))

puzzles = []
for r in rows:
    p = {"puzzle_id": r["puzzle_id"], "outcome": r["outcome"], "original": r["original_text"].strip()}
    if r["adapted_text"].strip():
        p["adapted"] = r["adapted_text"].strip()
    puzzles.append(p)

from collections import Counter
print(f"Loaded {len(puzzles)} puzzles:", dict(Counter(p["outcome"] for p in puzzles)))
has_adapted = sum(1 for p in puzzles if "adapted" in p)
print(f"{has_adapted} of {len(puzzles)} have usable adapted text (success + partial_fail);")
print(f"{len(puzzles) - has_adapted} are original-only (empty_fail — no parseable adaptation to compare against).")

## Min-K%-Prob via teacher-forced logprobs

In [ ]:
def min_k_prob(model_id: str, text: str, k_pct: int = K_PCT) -> float | None:
    """Score `text` itself (no generation) and return the mean log-prob of its
    lowest-k% tokens. Returns None if this Ollama version/model doesn't expose logprobs
    on /v1/completions — see the fallback note below if that happens."""
    resp = ollama_client.completions.create(
        model=model_id, prompt=text, max_tokens=0, echo=True, logprobs=1, temperature=0.0,
    )
    token_logprobs = resp.choices[0].logprobs.token_logprobs
    # first token has no preceding context, so its logprob is typically null — drop it
    vals = [lp for lp in token_logprobs if lp is not None]
    if not vals:
        return None
    k = max(1, int(len(vals) * k_pct / 100))
    lowest_k = sorted(vals)[:k]
    return sum(lowest_k) / len(lowest_k)

In [ ]:
# Quick capability check before running the full loop — if this errors or returns None,
# your Ollama build/model doesn't support logprobs on completions; see the fallback note below.
test_score = min_k_prob(MODELS[0], "This is a short test sentence for a logprob sanity check.")
print(f"Sanity check ({MODELS[0]}): {test_score}")
assert test_score is not None, "logprobs not returned — see fallback note in the markdown cell below"

**If the sanity check fails:** update Ollama to a recent version (`ollama --version`; logprobs support on
the OpenAI-compatible endpoint has been added incrementally). If it still doesn't return logprobs for a
particular model, the fallback is to run that model through `llama.cpp` or `transformers` directly and
pull `logits` from a single forward pass instead — more setup, only worth it if Ollama genuinely can't do it.

## Run: score original (all 21) and adapted (success + partial_fail only — 11) for each model

`empty_fail` puzzles have no `adapted` key, so they contribute an original-side score only — same pattern
as the Time Travel notebook.

In [ ]:
from tqdm import tqdm

records = []
for p in tqdm(puzzles, desc="puzzles"):
    text_types = ["original", "adapted"] if "adapted" in p else ["original"]
    for model_id in MODELS:
        for text_type in text_types:
            score = min_k_prob(model_id, p[text_type])
            records.append({
                "puzzle_id": p["puzzle_id"], "outcome": p["outcome"], "model": model_id,
                "text_type": text_type, "min_k_prob": score,
            })

results_df = pd.DataFrame(records)
results_df.to_csv("mink_prob_results.csv", index=False)
results_df.head()

## Compare: original vs. adapted, per model, split by outcome

In [ ]:
pivot = results_df.pivot_table(index=["puzzle_id", "outcome"], columns=["model", "text_type"], values="min_k_prob")
print(pivot.round(3))

summary = results_df.groupby(["model", "outcome", "text_type"])["min_k_prob"].agg(["mean", "median", "count"])
print()
print(summary.round(3))

In [ ]:
import matplotlib.pyplot as plt

OUTCOME_COLOR = {"success": "tab:blue", "partial_fail": "tab:orange"}

fig, axes = plt.subplots(1, len(MODELS), figsize=(5 * len(MODELS), 4), sharey=True)
if len(MODELS) == 1:
    axes = [axes]
for ax, model_id in zip(axes, MODELS):
    sub = results_df[results_df["model"] == model_id]
    orig = sub[sub["text_type"] == "original"].set_index("puzzle_id")
    adapt = sub[sub["text_type"] == "adapted"].set_index("puzzle_id")
    common = orig.index.intersection(adapt.index)   # only success + partial_fail have both
    for pid in common:
        color = OUTCOME_COLOR.get(orig.loc[pid, "outcome"], "gray")
        ax.plot([0, 1], [orig.loc[pid, "min_k_prob"], adapt.loc[pid, "min_k_prob"]],
                marker="o", alpha=0.7, color=color)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["original", "adapted"])
    ax.set_title(model_id); ax.set_ylabel("Min-K%-Prob (mean logprob, bottom 20% tokens)")
handles = [plt.Line2D([0], [0], color=c, marker="o", label=k) for k, c in OUTCOME_COLOR.items()]
axes[-1].legend(handles=handles, loc="best", fontsize=8)
plt.tight_layout()
plt.savefig("mink_prob_comparison.png", dpi=150)
plt.show()

## Reading the result

Each line is one puzzle: original on the left, adapted on the right — blue for `success`, orange for the
single `partial_fail` case. A pattern of lines sloping **downward** (original scores higher than adapted)
is the expected contamination signature — the model finds the original text more predictable than
genuinely novel text of matched complexity.

`empty_fail` puzzles have no adapted score to plot, but their **original** scores still belong in the
`summary` table above — they're 10 more data points on "does the model find GRE originals unusually
predictable," just without an adapted-side contrast.

The one `partial_fail` point is worth looking at on its own: if its drop from original to adapted is
similar in size to the `success` cases, that suggests the contamination signal tracks surface novelty
more than judged correctness — a coherent-but-wrong rewrite is already "novel enough." If it barely drops
at all, that's a different and also interesting finding. Either way, n=1 means this is one illustrative
data point, not a group comparison to generalize from.